In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "AVAXUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,vol_regime_ratio,hour_sin,hour_cos,dow_sin,dow_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,20.81,20.81,20.75,20.75,6791.48,2025-06-01 00:04:59.999999+00:00,141120.0255,753,3746.46,...,NaN,0.0,1.0,-0.781831,0.62349,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,20.76,20.79,20.76,20.78,4079.09,2025-06-01 00:09:59.999999+00:00,84697.1465,527,2504.22,...,NaN,0.0,1.0,-0.781831,0.62349,0.002393,0.000479,0.001915,NaN,NaN
2,2025-06-01 00:10:00+00:00,20.78,20.78,20.72,20.74,5606.32,2025-06-01 00:14:59.999999+00:00,116315.6147,478,682.15,...,NaN,0.0,1.0,-0.781831,0.62349,0.001050,0.000593,0.000457,NaN,NaN
3,2025-06-01 00:15:00+00:00,20.74,20.75,20.68,20.72,7006.76,2025-06-01 00:19:59.999999+00:00,145169.3124,626,4010.32,...,NaN,0.0,1.0,-0.781831,0.62349,-0.001610,0.000152,-0.001762,NaN,NaN
4,2025-06-01 00:20:00+00:00,20.71,20.74,20.68,20.72,5735.36,2025-06-01 00:24:59.999999+00:00,118814.0872,441,722.98,...,NaN,0.0,1.0,-0.781831,0.62349,-0.003675,-0.000613,-0.003062,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]
fwd_ret_train = train_df[ret_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]
fwd_ret_valid = valid_df[ret_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret_test = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,417
[info] optuna train rows: 53,386
[info] valid rows:        13,347
[info] test rows:         16,684


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fwd_ret_valid=fwd_ret_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-23 14:53:01,230] A new study created in memory with name: no-name-debf2c12-dfab-4be6-88ea-738798ce6060


[I 2026-03-23 14:53:05,556] Trial 0 finished with value: 0.5291067943211798 and parameters: {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 0.5, 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 0 with value: 0.5291067943211798.


[I 2026-03-23 14:53:14,099] Trial 1 finished with value: 0.518688284081217 and parameters: {'n_estimators': 500, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.8, 'bootstrap': False, 'class_weight': 'balanced_subsample', 'criterion': 'log_loss'}. Best is trial 0 with value: 0.5291067943211798.


[I 2026-03-23 14:53:17,649] Trial 2 finished with value: 0.531442829831332 and parameters: {'n_estimators': 800, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 2 with value: 0.531442829831332.


[I 2026-03-23 14:53:20,976] Trial 3 finished with value: 0.5328081340188285 and parameters: {'n_estimators': 700, 'max_depth': 12, 'min_samples_split': 11, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss'}. Best is trial 3 with value: 0.5328081340188285.


...

[I 2026-03-23 14:57:46,938] Trial 96 finished with value: 0.5352432945399033 and parameters: {'n_estimators': 800, 'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'gini'}. Best is trial 85 with value: 0.538094249565852.


[I 2026-03-23 14:57:51,851] Trial 97 finished with value: 0.5271208025081076 and parameters: {'n_estimators': 800, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.5, 'bootstrap': False, 'class_weight': None, 'criterion': 'gini'}. Best is trial 85 with value: 0.538094249565852.


[I 2026-03-23 14:57:53,732] Trial 98 finished with value: 0.5369104629532515 and parameters: {'n_estimators': 800, 'max_depth': 4, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'gini'}. Best is trial 85 with value: 0.538094249565852.


[I 2026-03-23 14:57:54,397] Trial 99 finished with value: 0.5346912371540493 and parameters: {'n_estimators': 200, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'criterion': 'gini'}. Best is trial 85 with value: 0.538094249565852.


['vol_30', 'atr_norm', 'imbalance_15', 'dist_ma_30', 'mom_60', 'vol_regime_ratio', 'mom_15', 'trend_strength', 'macd_hist', 'vol_5', 'vol_ratio_5_30', 'mom_5', 'dist_ma_15', 'bar_range', 'range_ratio', 'trades_z', 'num_trades_mom_5', 'volume_mom_5', 'volume_z', 'imbalance_z', 'co_spread', 'hour_cos', 'imbalance', 'taker_buy_ratio', 'hour_sin']
feature
vol_30              0.055579
atr_norm            0.050459
imbalance_15        0.050301
dist_ma_30          0.047838
mom_60              0.047213
vol_regime_ratio    0.046389
mom_15              0.044554
trend_strength      0.044109
macd_hist           0.043887
vol_5               0.039412
vol_ratio_5_30      0.038068
mom_5               0.037773
dist_ma_15          0.036112
bar_range           0.035957
range_ratio         0.035714
trades_z            0.032298
num_trades_mom_5    0.031437
volume_mom_5        0.031422
volume_z            0.030854
imbalance_z         0.030585
co_spread           0.029907
hour_cos            0.029132
imbalanc

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

In [11]:
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

train_pred = base_model.predict_proba(X_train_full_sel)[:, 1]
test_pred = base_model.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_ic = spearmanr(train_pred, fwd_ret_train)[0]
test_ic = spearmanr(test_pred, fwd_ret_test)[0]

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train IC:        {train_ic:.6f}")
print(f"Test IC:         {test_ic:.6f}")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:        0.127668
Test IC:         0.047215
Train ROC AUC:   0.583539
Test ROC AUC:    0.537680
Train PR AUC:    0.559313
Test PR AUC:     0.469775
Train Log Loss:  0.685171
Test Log Loss:   0.685824
Train Brier:     0.246028
Test Brier:      0.246348
Train Accuracy:  0.551242
Test Accuracy:   0.561616
Train Precision: 0.692796
Test Precision:  0.496009
Train Recall:    0.070053
Test Recall:     0.059532
Train F1:        0.127240
Test F1:         0.106305


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret_test": fwd_ret_test.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.405, 0.443] -0.000282   1669  0.004480
(0.443, 0.451]  0.000075   1668  0.005248
(0.451, 0.457] -0.000139   1668  0.005373
(0.457, 0.465] -0.000138   1669  0.005953
(0.465, 0.473] -0.000441   1668  0.005803
(0.473, 0.479] -0.000029   1668  0.005731
(0.479, 0.483]  0.000050   1669  0.005721
(0.483, 0.487] -0.000001   1668  0.005650
(0.487, 0.492]  0.000143   1668  0.005818
(0.492, 0.64]  -0.000042   1669  0.010076


/tmp/ipykernel_1424412/3344132490.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret_test"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret_test"].mean())
overall_mean_ret = float(eval_df["fwd_ret_test"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret_test"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/AVAXUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": float(train_ic),
    "test_ic": float(test_ic),
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/AVAXUSDT__h6_model.joblib
[saved] features -> models/rf/AVAXUSDT__h6_feature_cols.json
[saved] feature importance -> models/rf/AVAXUSDT__h6_feature_importance.csv
[saved] metadata -> models/rf/AVAXUSDT__h6_meta.json
